# UDACITY SageMaker Essentials: Tying it all Together

## Using XGBoost in SageMaker


To tie everything together, we'll be using Amazon's SageMaker service to construct a random tree model to predict the sentiment of a movie review. This is also a sentiment analysis problem but we won't be using BlazingText. Instead, we will be using the XGBoost package as it is provided to us by Amazon.

## Instructions
Once you this notebook is opened in SageMaker Studio, select "Python3(Data Science)" kernel with "ml.g4dn.xlarge" instance.

Some template code has already been provided for you, and you will need to implement additional functionality to successfully complete this notebook. You will not need to modify the included code beyond what is requested. Sections that begin with '**TODO**' in the header indicate that you need to complete or implement some portion within them. Instructions will be provided for each section and the specifics of the implementation are marked in the code block with a `# TODO: ...` comment. Please be sure to read the instructions carefully!

## Step 1: Downloading the data

The dataset we are going to use is very popular among researchers in Natural Language Processing, usually referred to as the [IMDb dataset](http://ai.stanford.edu/~amaas/data/sentiment/). It consists of movie reviews from the website [imdb.com](http://www.imdb.com/), each labeled as either '**pos**itive', if the reviewer enjoyed the film, or '**neg**ative' otherwise.

> Maas, Andrew L., et al. [Learning Word Vectors for Sentiment Analysis](http://ai.stanford.edu/~amaas/data/sentiment/). In _Proceedings of the 49th Annual Meeting of the Association for Computational Linguistics: Human Language Technologies_. Association for Computational Linguistics, 2011.

We begin by using some Jupyter Notebook magic to download and extract the dataset.

In [1]:
%mkdir ../data
!wget -O ../data/aclImdb_v1.tar.gz https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -zxf ../data/aclImdb_v1.tar.gz -C ../data

mkdir: cannot create directory ‘../data’: File exists
--2026-09-29 15:00:05--  https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘../data/aclImdb_v1.tar.gz’

../data/aclImdb_v1. 100%[===================>]  80.23M  22.5MB/s    in 3.6s    

2026-09-29 15:00:09 (22.5 MB/s) - ‘../data/aclImdb_v1.tar.gz’ saved [84125825/84125825]



## Step 2: Preparing the data

The data we have downloaded is split into various files, each of which contains a single review. It will be much easier going forward if we combine these individual files into two large files, one for training and one for testing.

In [3]:
import os
import glob

def read_imdb_data(data_dir='../data/aclImdb'):
    data = {}
    labels = {}
    
    # Full dataset is large; modify these limits as appropriate
    TRAIN_LIMIT = 12500 # 10000
    TEST_LIMIT= 1000
    
    for data_type in ['train', 'test']:
        data[data_type] = {}
        labels[data_type] = {}
        
        for sentiment in ['pos', 'neg']:
            data[data_type][sentiment] = []
            labels[data_type][sentiment] = []
            
            path = os.path.join(data_dir, data_type, sentiment, '*.txt')
            files = glob.glob(path)
            
            file_index = 0
            for f in files:
                if file_index >= TRAIN_LIMIT and data_type == 'train': break
                if file_index >= TEST_LIMIT and data_type == 'test': break
                file_index += 1
                with open(f) as review:
                    data[data_type][sentiment].append(review.read())
                    # Here we represent a positive review by '1' and a negative review by '0'
                    labels[data_type][sentiment].append(1 if sentiment == 'pos' else 0)
                    
            assert len(data[data_type][sentiment]) == len(labels[data_type][sentiment]), \
                    "{}/{} data size does not match labels size".format(data_type, sentiment)
                
    return data, labels

In [4]:
data, labels = read_imdb_data()
print("IMDB reviews: train = {} pos / {} neg, test = {} pos / {} neg".format(
            len(data['train']['pos']), len(data['train']['neg']),
            len(data['test']['pos']), len(data['test']['neg'])))

IMDB reviews: train = 12500 pos / 12500 neg, test = 1000 pos / 1000 neg


In [5]:
from sklearn.utils import shuffle

def prepare_imdb_data(data, labels):
    """Prepare training and test sets from IMDb movie reviews."""
    
    #Combine positive and negative reviews and labels
    data_train = data['train']['pos'] + data['train']['neg']
    data_test = data['test']['pos'] + data['test']['neg']
    labels_train = labels['train']['pos'] + labels['train']['neg']
    labels_test = labels['test']['pos'] + labels['test']['neg']
    
    #Shuffle reviews and corresponding labels within training and test sets
    data_train, labels_train = shuffle(data_train, labels_train)
    data_test, labels_test = shuffle(data_test, labels_test)
    
    # Return a unified training data, test data, training labels, test labets
    return data_train, data_test, labels_train, labels_test

In [6]:
train_X, test_X, train_y, test_y = prepare_imdb_data(data, labels)
print("IMDb reviews (combined): train = {}, test = {}".format(len(train_X), len(test_X)))

IMDb reviews (combined): train = 25000, test = 2000


In [7]:
train_X[100]

"If this could be rated a 0, it would be. From the atrocious wooden acting to the dull, plodding pace, the puerile exploitation angle and the sophomoric pseudolesbian titillation added purely for the sake of the sad viewers, this is a disaster of a film.<br /><br />The plot is predictable, as from the beginning one is absolutely certain of what will happen to all of the characters and how the plot will commence. From a boring, unexceptional beginning to a pathetic and bleak ending, every single presence in this film is wasted, and every meagre scrap of talent dug up for this turkey is squandered.<br /><br />If you want to watch something as relentlessly bleak with plenty of the same childish titillation, watch one of the Ilsa films. At least they're unapologetic and up-front about what they're trying to do. How something this horrendously bad ever managed to be rated above a 5 is a miracle of how people with no taste or discerning standards can sometimes come together for the most dubi

## Step 3: Processing the data

Now that we have our training and testing datasets merged and ready to use, we need to start processing the raw data into something that will be useable by our machine learning algorithm. To begin with, we remove any html formatting that may appear in the reviews and perform some standard natural language processing in order to homogenize the data.

In [8]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem.porter import *
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
import re
from bs4 import BeautifulSoup

def review_to_words(review):
    text = BeautifulSoup(review, "html.parser").get_text() # Remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9]", " ", text.lower()) # Convert to lower case
    words = text.split() # Split string into words
    words = [w for w in words if w not in stopwords.words("english")] # Remove stopwords
    # words = [PorterStemmer().stem(w) for w in words] # stem
    words = [stemmer.stem(w) for w in words]
    
    return words

In [10]:
import pickle
from tqdm import tqdm

cache_dir = os.path.join("../cache", "sentiment_analysis")  # where to store cache files
os.makedirs(cache_dir, exist_ok=True)  # ensure cache directory exists

def preprocess_data(data_train, data_test, labels_train, labels_test,
                    cache_dir=cache_dir, cache_file="preprocessed_data.pkl"):
    """Convert each review to words; read from cache if available."""

    # If cache_file is not None, try to read from it first
    cache_data = None
    if cache_file is not None:
        try:
            with open(os.path.join(cache_dir, cache_file), "rb") as f:
                cache_data = pickle.load(f)
            print("Read preprocessed data from cache file:", cache_file)
        except:
            pass  # unable to read from cache, but that's okay
    
    # If cache is missing, then do the heavy lifting
    if cache_data is None:
        # Preprocess training and test data to obtain words for each review
        #words_train = list(map(review_to_words, data_train))
        #words_test = list(map(review_to_words, data_test))
        # words_train = [review_to_words(review) for review in data_train]
        # words_test = [review_to_words(review) for review in data_test]
        words_train = [review_to_words(r) for r in tqdm(data_train)]
        words_test = [review_to_words(r) for r in tqdm(data_test)]

        
        # Write to cache file for future runs
        if cache_file is not None:
            cache_data = dict(words_train=words_train, words_test=words_test,
                              labels_train=labels_train, labels_test=labels_test)
            with open(os.path.join(cache_dir, cache_file), "wb") as f:
                pickle.dump(cache_data, f)
            print("Wrote preprocessed data to cache file:", cache_file)
    else:
        # Unpack data loaded from cache file
        words_train, words_test, labels_train, labels_test = (cache_data['words_train'],
                cache_data['words_test'], cache_data['labels_train'], cache_data['labels_test'])
    
    return words_train, words_test, labels_train, labels_test

In [11]:
# Preprocess data
train_X, test_X, train_y, test_y = preprocess_data(train_X, test_X, train_y, test_y)

Read preprocessed data from cache file: preprocessed_data.pkl


### Extract Bag-of-Words features

For the model we will be implementing, rather than using the reviews directly, we are going to transform each review into a Bag-of-Words feature representation. Keep in mind that 'in the wild' we will only have access to the training set so our transformer can only use the training set to construct a representation.

In [12]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import joblib
# joblib is an enhanced version of pickle that is more efficient for storing NumPy arrays

def extract_BoW_features(words_train, words_test, vocabulary_size=5000,
                         cache_dir=cache_dir, cache_file="bow_features.pkl"):
    """Extract Bag-of-Words for a given set of documents, already preprocessed into words."""
    
    # If cache_file is not None, try to read from it first
    cache_data = None
    if cache_file is not None:
        try:
            with open(os.path.join(cache_dir, cache_file), "rb") as f:
                cache_data = joblib.load(f)
            print("Read features from cache file:", cache_file)
        except:
            pass  # unable to read from cache, but that's okay
    
    # If cache is missing, then do the heavy lifting
    if cache_data is None:
        # Fit a vectorizer to training documents and use it to transform them
        # NOTE: Training documents have already been preprocessed and tokenized into words;
        #       pass in dummy functions to skip those steps, e.g. preprocessor=lambda x: x
        vectorizer = CountVectorizer(max_features=vocabulary_size,
                preprocessor=lambda x: x, tokenizer=lambda x: x)  # already preprocessed
        features_train = vectorizer.fit_transform(words_train).toarray()

        # Apply the same vectorizer to transform the test documents (ignore unknown words)
        features_test = vectorizer.transform(words_test).toarray()
        
        # NOTE: Remember to convert the features using .toarray() for a compact representation
        
        # Write to cache file for future runs (store vocabulary as well)
        if cache_file is not None:
            vocabulary = vectorizer.vocabulary_
            cache_data = dict(features_train=features_train, features_test=features_test,
                             vocabulary=vocabulary)
            with open(os.path.join(cache_dir, cache_file), "wb") as f:
                joblib.dump(cache_data, f)
            print("Wrote features to cache file:", cache_file)
    else:
        # Unpack data loaded from cache file
        features_train, features_test, vocabulary = (cache_data['features_train'],
                cache_data['features_test'], cache_data['vocabulary'])
    
    # Return both the extracted features as well as the vocabulary
    return features_train, features_test, vocabulary

In [13]:
# Extract Bag of Words features for both training and test datasets
train_X, test_X, vocabulary = extract_BoW_features(train_X, test_X)

Read features from cache file: bow_features.pkl


## Step 4: Classification using XGBoost

Now that we have created the feature representation of our training (and testing) data, it is time to start setting up and using the XGBoost classifier provided by SageMaker.

### (TODO) Writing the dataset

The XGBoost classifier that we will be using requires the dataset to be written to a file and stored using Amazon S3. To do this, we will start by splitting the training dataset into two parts, the data we will train the model with and a validation set. Then, we will write those datasets to a file and upload the files to S3. In addition, we will write the test set input to a file and upload the file to S3. This is so that we can use SageMakers Batch Transform functionality to test our model once we've fit it.

In [12]:
import pandas as pd

# TODO: Split the train_X and train_y arrays into the DataFrames val_X, train_X and val_y, train_y. 
# Make sure that val_X and val_y contain 10000 entires while train_X and train_y contain the remaining 15000 entries.
val_X = pd.DataFrame(train_X[:10000])
train_X = pd.DataFrame(train_X[10000:])

val_y = pd.DataFrame(train_y[:10000])
train_y = pd.DataFrame(train_y[10000:])

The documentation for the XGBoost algorithm in SageMaker requires that the saved datasets should contain no headers or index and that for the training and validation data, the label should occur first for each sample.

For more information about this and other algorithms, the SageMaker developer documentation can be found on __[Amazon's website.](https://docs.aws.amazon.com/sagemaker/latest/dg/)__

In [17]:
import os
# First we make sure that the local directory in which we'd like to store the training and validation csv files exists.
data_dir = 'data/xgboost'
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

In [14]:
# Then, save the test data to test.csv in the data_dir directory. Note that we do not save the associated ground truth
# labels, instead we will use them later to compare with our model output.

pd.DataFrame(test_X).to_csv(os.path.join(data_dir, 'test.csv'), header=False, index=False)

# TODO: Save the training and validation data to train.csv and validation.csv in the data_dir directory.
# Make sure that the files you create include the ground truth and are in the correct format.

#Insert Label column for train_X and val_X
train_X.insert(0, "label", train_y)
val_X.insert(0, "label", val_y)

In [15]:
train_X.head()

,label,0,1,2,3,4,5,6,7,8,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,1,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
# Save train_X an val_X to CSV
train_X.to_csv(os.path.join(data_dir, 'train.csv'), header=False, index=False)
val_X.to_csv(os.path.join(data_dir, 'validation.csv'), header=False, index=False)

In [17]:
# To save a bit of memory we can set text_X, train_X, val_X, train_y and val_y to None.

test_X = train_X = val_X = train_y = val_y = None

### (TODO) Uploading Training / Validation files to S3

Amazon's S3 service allows us to store files that can be access by both the built-in training models such as the XGBoost model we will be using as well as custom models such as the one we will see a little later.

For this, and most other tasks we will be doing using SageMaker, there are two methods we could use. The first is to use the low level functionality of SageMaker which requires knowing each of the objects involved in the SageMaker environment. The second is to use the high level functionality in which certain choices have been made on the user's behalf. The low level approach benefits from allowing the user a great deal of flexibility while the high level approach makes development much quicker. For our purposes we will opt to use the high level approach although using the low-level approach is certainly an option.

The method `upload_data()`, a member of the object representing our current SageMaker session, may be of use to you. What this method does is upload the data to the default bucket (which is created if it does not exist) into the path described by the key_prefix variable. To see this for yourself, once you have uploaded the data files, go to the S3 console and look to see where the files have been uploaded.

For additional resources, see the __[SageMaker session.upload_data](https://sagemaker.readthedocs.io/en/latest/session.html#sagemaker.session.Session.upload_data)__ and in addition the __[SageMaker Developer Guide.](https://docs.aws.amazon.com/sagemaker/latest/dg/)__

In [19]:
# Adding this code block to upload the files since the existing suggested method in the exercice is deprecated
import logging
import boto3
from botocore.exceptions import ClientError
import os


def upload_file(file_name, bucket, object_name=None):
    """Upload a file to an S3 bucket

    :param file_name: File to upload
    :param bucket: Bucket to upload to
    :param object_name: S3 object name. If not specified then file_name is used
    :return: True if file was uploaded, else False
    """

    # If S3 object_name was not specified, use file_name
    if object_name is None:
        object_name = os.path.basename(file_name)

    # Upload the file
    s3_client = boto3.client('s3')
    try:
        response = s3_client.upload_file(file_name, bucket, object_name)
    except ClientError as e:
        logging.error(e)
        return False
    return True

In [9]:
import sagemaker

# session = sagemaker.Session() # Store the current SageMaker session <- deprecated


# Bucket
bucket = "sagemaker-us-east-1-864107284170"

# S3 prefix (which folder will we use)
prefix = 'lesson2/ex5/sentiment-xgboost'

# TODO: Upload the test.csv, train.csv and validation.csv files which are contained in data_dir to S3 using sess.upload_data().
test_location = f"{prefix}/test.csv"
val_location = f"{prefix}/validation.csv"
train_location = f"{prefix}/train.csv"

upload_file(os.path.join(data_dir, 'test.csv'), bucket, test_location)
upload_file(os.path.join(data_dir, 'validation.csv'), bucket, val_location)
upload_file(os.path.join(data_dir, 'train.csv'), bucket, train_location)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:17                                                                                   │
│                                                                                                  │
│   14 val_location = f"{prefix}/validation.csv"                                                   │
│   15 train_location = f"{prefix}/train.csv"                                                      │
│   16                                                                                             │
│ ❱ 17 upload_file(os.path.join(data_dir, 'test.csv'), bucket, test_location)                      │
│   18 upload_file(os.path.join(data_dir, 'validation.csv'), bucket, val_location)                 │
│   19 upload_file(os.path.join(data_dir, 'train.csv'), bucket, train_location)                    │
│   20                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'upload_file' is not defined

### (TODO) Creating the XGBoost model

Now that the data has been uploaded it is time to create the XGBoost model. To begin with, we need to do some setup. At this point it is worth discussing what a model is in SageMaker. It is easiest to think of a model of comprising three different objects in the SageMaker ecosystem, which interact with one another.

- Model Artifacts
- Training Code (Container)
- Inference Code (Container)

The Model Artifacts are what you might think of as the actual model itself. For example, if you were building a neural network, the model artifacts would be the weights of the various layers. In our case, for an XGBoost model, the artifacts are the actual trees that are created during training.

The other two objects, the training code and the inference code are then used the manipulate the training artifacts. More precisely, the training code uses the training data that is provided and creates the model artifacts, while the inference code uses the model artifacts to make predictions on new data.

The way that SageMaker runs the training and inference code is by making use of Docker containers. For now, think of a container as being a way of packaging code up so that dependencies aren't an issue.

In [10]:
# from sagemaker import get_execution_role <- deprecated
from sagemaker.core.helper.session_helper import get_execution_role

# Our current execution role is require when creating the model as the training
# and inference code will need to access the model artifacts.
role = get_execution_role()

In [11]:
# We need to retrieve the location of the container which is provided by Amazon for using XGBoost.
# As a matter of convenience, the training and inference code both use the same container.

# from sagemaker import image_uris <- deprecated
from sagemaker.core import image_uris

# container = image_uris.retrieve('xgboost', session.boto_region_name, 'latest') <- deprecated

container = image_uris.retrieve(framework="xgboost", version="latest", region="us-east-1")

[09/30/26 09:57:25] INFO     Ignoring unnecessary instance type: None.                            ]8;id=1825627;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=1825628;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

In [13]:
# TODO: Create a SageMaker estimator using the container location determined in the previous cell.
#       It is recommended that you use a single training instance of type ml.m4.xlarge. It is also
#       recommended that you use 's3://{}/{}/output'.format(session.default_bucket(), prefix) as the
#       output path.

# TODO: Set the XGBoost hyperparameters in the xgb object. Don't forget that in this case we have a binary
#       label so we should be using the 'binary:logistic' objective.

from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import Compute, OutputDataConfig, InputData
from sagemaker.core.training.configs import StoppingCondition


xgb = ModelTrainer(
    training_image=container,
    role=role,
    compute=Compute(
        instance_type="ml.m5.xlarge",
        instance_count=1,
    ),
    output_data_config=OutputDataConfig(
        s3_output_path=f"s3://{bucket}/{prefix}/output",
    ),
    hyperparameters={
        "max_depth": "5",
        "objective": "binary:logistic",
        "early_stopping_rounds": "10",
        "num_round": "200",
    },
    stopping_condition=StoppingCondition(max_runtime_in_seconds=900)
)


[09/30/26 10:04:01] INFO     SageMaker session not provided. Using default Session.                  ]8;id=1825670;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=1825671;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Base name not provided. Using default name:                             ]8;id=1825676;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=1825677;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#90\90]8;;\
                             xgboost-job                                                                           

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=1825682;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=1825683;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=1825688;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=1825689;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             811284229777.dkr.ecr.us-east-1.amazonaws.com/xgboost:latest                           

### Fit the XGBoost model

Now that our model has been set up we simply need to attach the training and validation datasets and then ask SageMaker to set up the computation.

In [16]:

# s3_input_train = sagemaker.TrainingInput(s3_data=train_location, content_type='csv') <- deprecated
# s3_input_validation = sagemaker.TrainingInput(s3_data=val_location, content_type='csv') <- deprecated

s3_input_train = InputData(
    channel_name="train",
    data_source=f"s3://{bucket}/{prefix}/train.csv",
    content_type="text/csv",
)

s3_input_validation = InputData(
    channel_name="validation",
    data_source=f"s3://{bucket}/{prefix}/validation.csv",
    content_type="text/csv",
)

In [17]:
# xgb.fit({'train': s3_input_train, 'validation': s3_input_validation}) <- deprecated

xgb.train(
    input_data_config=[s3_input_train, s3_input_validation],
    wait=True
)

[09/30/26 10:29:50] INFO     Creating training_job resource.                                     ]8;id=1825770;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1825771;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31238\31238]8;;\

Output()

[09/30/26 10:31:57] INFO     xgboost-job-20260930102950/algo-1-1790764238:                       ]8;id=1825776;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1825777;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Arguments: train                                                                      

                    INFO     xgboost-job-20260930102950/algo-1-1790764238:                       ]8;id=1825782;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1825783;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [2026-09-30:10:31:50:INFO] Running standalone xgboost training.                       

                    INFO     xgboost-job-20260930102950/algo-1-1790764238:                       ]8;id=1825788;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1825789;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [2026-09-30:10:31:50:INFO] File size need to be processed in the                      
                             node: 238.47mb. Available memory size in the node: 8023.39mb                          

...

                    INFO     xgboost-job-20260930102950/algo-1-1790764238:                       ]8;id=1827330;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827331;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [10:33:46] src/tree/updater_prune.cc:74: tree pruning end, 1 roots,                   
                             36 extra nodes, 0 pruned nodes, max_depth=5                                           

                    INFO     xgboost-job-20260930102950/algo-1-1790764238:                       ]8;id=1827336;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827337;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [124]#011train-error:0.0514#011validation-error:0.1451                                

                    INFO     xgboost-job-20260930102950/algo-1-1790764238:                       ]8;id=1827342;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827343;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Stopping. Best iteration:                                                             

                    INFO     xgboost-job-20260930102950/algo-1-1790764238:                       ]8;id=1827348;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827349;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [114]#011train-error:0.054933#011validation-error:0.1444                              

[09/30/26 10:34:04] INFO     Final Resource Status: Completed                                    ]8;id=1827354;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827355;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31590\31590]8;;\

### (TODO) Testing the model

Now that we've fit our XGBoost model, it's time to see how well it performs. To do this we will use SageMakers Batch Transform functionality. Batch Transform is a convenient way to perform inference on a large dataset in a way that is not realtime. That is, we don't necessarily need to use our model's results immediately and instead we can peform inference on a large number of samples. An example of this in industry might be peforming an end of month report. This method of inference can also be useful to us as it means to can perform inference on our entire test set. 

To perform a Batch Transformation we need to first create a transformer objects from our trained estimator object.

In [19]:
# # TODO: Create a transformer object from the trained model. 
# # Using an instance count of 1 and an instance type of ml.m4.xlarge should be more than enough.
# xgb_transformer = None

# The estimator's .transformer() shortcut is gone. We should use the TransformJob object in sagemaker v3 api. 
# First, we must register the artifact as a Model

from sagemaker.core.resources import Model
from sagemaker.core.shapes import ContainerDefinition

model_data = xgb._latest_training_job.model_artifacts.s3_model_artifacts

Model.create(
    model_name="sentiment-xgboost",
    primary_container=ContainerDefinition(
        image=container,
        model_data_url=model_data,
    ),
    execution_role_arn=role,
    region="us-east-1",
)

[09/30/26 11:23:15] INFO     Creating model resource.                                            ]8;id=1827361;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827362;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#20592\20592]8;;\

Model(model_name='sentiment-xgboost', primary_container=ContainerDefinition(container_hostname=Unassigned(), image='811284229777.dkr.ecr.us-east-1.amazonaws.com/xgboost:latest', image_config=Unassigned(), mode='SingleModel', model_data_url='s3://sagemaker-us-east-1-864107284170/lesson2/ex5/sentiment-xgboost/output/xgboost-job-20260930102950/output/model.tar.gz', model_data_source=ModelDataSource(s3_data_source=S3ModelDataSource(s3_data_type='S3Object', compression_type='Gzip', s3_uri='s3://sagemaker-us-east-1-864107284170/lesson2/ex5/sentiment-xgboost/output/xgboost-job-20260930102950/output/model.tar.gz', model_access_config=Unassigned(), hub_access_config=Unassigned(), manifest_s3_uri=Unassigned(), e_tag=Unassigned(), manifest_etag=Unassigned())), additional_model_data_sources=Unassigned(), environment=Unassigned(), model_package_name=Unassigned(), inference_specification_name=Unassigned(), multi_model_config=Unassigned()), containers=Unassigned(), inference_execution_config=Unassign

Next we actually perform the transform job. When doing so we need to make sure to specify the type of data we are sending so that it is serialized correctly in the background. In our case we are providing our model with csv data so we specify `text/csv`. Also, if the test data that we have provided is too large to process all at once then we need to specify how the data file should be split up. Since each line is a single entry in our data set we tell SageMaker that it can split the input on each line.

In [ ]:
from sagemaker.core.transformer import Transformer

# TODO: Start the transform job. Make sure to specify the content type and the split type of the test data.

transformer = Transformer(
    model_name = "sentiment-xgboost",
    instance_type="ml.m5.xlarge",
    instance_count = 1,
    output_path=f"s3://{bucket}/{prefix}/output/tranform-output/",
)

transformer.transform(
    data=f"s3://{bucket}/{prefix}/test.csv",
    data_type="S3Prefix",
    content_type="text/csv", 
    split_type="Line"
)


Currently the transform job is running but it is doing so in the background. Since we wish to wait until the transform job is done and we would like a bit of feedback we can run the `wait()` method.

In [25]:
# xgb_transformer.wait()

transformer.wait()

Output()

[09/30/26 11:47:47] INFO     sentiment-xgboost-2026-09-30-11-41-24-539/i-039b5fddc742fefd0-17907 ]8;id=1827683;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827684;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             68617:                                                                                
                             Arguments: serve                                                                      

                    INFO     sentiment-xgboost-2026-09-30-11-41-24-539/i-039b5fddc742fefd0-17907 ]8;id=1827689;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827690;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             68617:                                                                                
                             [2026-09-30 11:45:15 +0000] [1] [INFO] Starting gunicorn 19.9.0                       

                    INFO     sentiment-xgboost-2026-09-30-11-41-24-539/i-039b5fddc742fefd0-17907 ]8;id=1827695;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827696;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             68617:                                                                                
                             [2026-09-30 11:45:15 +0000] [1] [INFO] Listening at:                                  
                             http://0.0.0.0:8080 (1)                                                               

                    INFO     sentiment-xgboost-2026-09-30-11-41-24-539/i-039b5fddc742fefd0-17907 ]8;id=1827701;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827702;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             68617:                                                                                
                             [2026-09-30 11:45:15 +0000] [1] [INFO] Using worker: gevent                           

...

                    INFO     sentiment-xgboost-2026-09-30-11-41-24-539/i-039b5fddc742fefd0-17907 ]8;id=1827809;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827810;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             68617:                                                                                
                             [2026-09-30:11:45:19:INFO] Determined delimiter of CSV input is ','                   

                    INFO     sentiment-xgboost-2026-09-30-11-41-24-539/i-039b5fddc742fefd0-17907 ]8;id=1827815;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827816;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             68617:                                                                                
                             [2026-09-30:11:45:19:INFO] Sniff delimiter as ','                                     

                    INFO     sentiment-xgboost-2026-09-30-11-41-24-539/i-039b5fddc742fefd0-17907 ]8;id=1827821;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827822;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             68617:                                                                                
                             [2026-09-30:11:45:19:INFO] Determined delimiter of CSV input is ','                   

                    INFO     sentiment-xgboost-2026-09-30-11-41-24-539/i-039b5fddc742fefd0-17907 ]8;id=1827827;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827828;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             68617/data-log:                                                                       
                             2026-09-30T11:45:17.910:[sagemaker logs]:                                             
                             MaxConcurrentTransforms=4, MaxPayloadInMB=6,                                          
                             BatchStrategy=MULTI_RECORD                                                            

                    INFO     Final Resource Status: Completed                                    ]8;id=1827833;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1827834;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32482\32482]8;;\

Now the transform job has executed and the result, the estimated sentiment of each review, has been saved on S3. Since we would rather work on this file locally we can perform a bit of notebook magic to copy the file to the `data_dir`.

In [28]:
# !aws s3 cp --recursive $xgb_transformer.output_path $data_dir
output_path = transformer.output_path
print(output_path)

s3://sagemaker-us-east-1-864107284170/lesson2/ex5/sentiment-xgboost/output/tranform-output/


In [31]:
!aws s3 cp --recursive {output_path} {data_dir}

download: s3://sagemaker-us-east-1-864107284170/lesson2/ex5/sentiment-xgboost/output/tranform-output/test.csv.out to data/xgboost/test.csv.out


The last step is now to read in the output from our model, convert the output to something a little more usable, in this case we want the sentiment to be either `1` (positive) or `0` (negative), and then compare to the ground truth labels.

In [18]:
predictions = pd.read_csv(os.path.join(data_dir, 'test.csv.out'), header=None)
predictions = [round(num) for num in predictions.squeeze().values]

In [19]:
from sklearn.metrics import accuracy_score
accuracy_score(test_y, predictions)

0.857

## Optional: Clean up


In [21]:
# First we will remove all of the files contained in the data_dir directory
!rm $data_dir/*

# And then we delete the directory itself
!rmdir $data_dir

# Similarly we will remove the files in the cache_dir directory and the directory itself
!rm $cache_dir/*
!rmdir $cache_dir

rm: cannot remove 'data/xgboost/*': No such file or directory
rmdir: failed to remove 'data/xgboost': Directory not empty
rm: cannot remove '../cache/sentiment_analysis/*': No such file or directory
rmdir: failed to remove '../cache/sentiment_analysis': No such file or directory
